In [9]:
import copy

import numpy as np
import pandas as pd
import anndata as ad

In [10]:

import torch
import torch.nn as nn
import torch.optim as optim

### Load Data

In [11]:
input_dir = "/Users/joshuayang/Desktop/KB/data"
adata_train = ad.read_h5ad(input_dir+'/Shaffer_cancer/shaffer_train.h5ad')
adata_test = ad.read_h5ad(input_dir+'/Shaffer_cancer/shaffer_test.h5ad')


train_labels = adata_train.obs["clone_id"].to_numpy()
test_labels = adata_test.obs["clone_id"].to_numpy()

# embed_dir = input_dir + "/feat_LCL_2025/shaffer_cancer/feat_shaffer_lambda005_unlab5_bs110_testAsPenalty_best"
# X_train  = np.load(embed_dir+'/train_proj_embed.npy')
# X_test = np.load(embed_dir+'/test_proj_embed.npy')

embed_dir = input_dir + "/feat_LCL_2025/shaffer_cancer/feat_shaffer_lambda01_unlab5_bs100"
X_train  = np.load(embed_dir+'/train_base_embed.npy')
X_test = np.load(embed_dir+'/test_base_embed.npy')


print(adata_train.shape, adata_test.shape)
print(X_train.shape, X_test.shape)

(20656, 2000) (2368, 2000)
(20656, 64) (2368, 64)


In [12]:
adata_train.obs

,orig.ident,nCount_RNA,nFeature_RNA,nCount_lineage,nFeature_lineage,percent.mt,percent.rb,S.Score,G2M.Score,Phase,RNA_snn_res.0.5,seurat_clusters,RNA_snn_res.0.4,OG_condition,RNA_snn_res.0.3,Lineage,keep,clone_id,OG_condition_name
cistodabtram_CCCGGAAAGCAACTCT-1,SeuratProject,14914.0,4051,29.0,3,6.108355,9.145769,-0.003093,-0.154415,0,0,8,10,3,8,349,1,349,cistodabtram
cistodabtram_CATTCATAGCTAATGA-1,SeuratProject,4243.0,2031,24.0,3,7.612538,7.164742,-0.071747,-0.145342,0,8,7,7,3,7,349,1,349,cistodabtram
cistodabtram_CCTCAGTTCCTCTTTC-1,SeuratProject,24000.0,5311,43.0,5,12.800000,7.000000,-0.059770,0.227976,1,0,8,10,3,8,349,1,349,cistodabtram
cis_ACGATGTGTCGCGTTG-1,SeuratProject,10600.0,3288,8.0,3,11.943396,8.500000,-0.025121,0.107649,1,7,3,8,0,3,349,1,349,cis
cistodabtram_TTCATGTAGGGAGATA-1,SeuratProject,7200.0,2709,27.0,1,6.833333,7.611111,0.517406,-0.041500,2,1,8,10,3,8,349,1,349,cistodabtram
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
cocl2_TACGCTCAGCATGCAG-1,SeuratProject,6434.0,2193,33.0,2,6.310227,12.449487,-0.141095,-0.199297,0,8,7,7,4,7,464,1,464,cocl2
cocl2tocis_AAACCCACAATCCTAG-1,SeuratProject,7333.0,2816,49.0,8,12.491477,7.623074,0.076776,-0.172086,2,0,1,2,5,1,464,1,464,cocl2tocis
cocl2tocis_AGTGACTCAACCCGCA-1,SeuratProject,8566.0,2979,37.0,12,15.958440,8.452020,0.398226,-0.233381,2,5,3,6,5,3,464,1,464,cocl2tocis
cocl2tocis_GTGGGAATCTACAGGT-1,SeuratProject,5345.0,2324,70.0,9,12.029935,7.988775,-0.061381,0.571929,1,6,3,8,5,3,464,1,464,cocl2tocis


In [13]:
# -----------------------
# 1) Linear decoder
# -----------------------
class LinearSoftmax(nn.Module):
    def __init__(self, input_size, output_size):
        super().__init__()
        self.fc = nn.Linear(input_size, output_size)

    def forward(self, x):
        return self.fc(x)  # logits


# -----------------------
# 2) Train with early stopping
# -----------------------
def train_kl_earlystop(
    model,
    X_train,
    y_train,
    lr=5e-3,
    weight_decay=1e-4,
    max_epochs=5000,
    batch_size=256,
    val_frac=0.2,
    patience=150,
    min_delta=1e-5,
    seed=42,
    device=None,
    print_every=50,
    verbose=True,
):
    if device is None:
        if torch.cuda.is_available():
            device = "cuda"
        elif torch.backends.mps.is_available():
            device = "mps"
        else:
            device = "cpu"

    model = model.to(device)
    X_train = X_train.to(device)
    y_train = y_train.to(device)

    n = X_train.shape[0]
    g = torch.Generator(device="cpu").manual_seed(seed)
    perm = torch.randperm(n, generator=g)

    n_val = int(round(val_frac * n))
    val_idx = perm[:n_val]
    tr_idx = perm[n_val:]

    X_tr, y_tr = X_train[tr_idx], y_train[tr_idx]
    X_val, y_val = X_train[val_idx], y_train[val_idx]

    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    criterion = nn.KLDivLoss(reduction="batchmean")

    best_val = float("inf")
    best_state = copy.deepcopy(model.state_dict())
    best_epoch = -1
    bad_epochs = 0

    @torch.no_grad()
    def eval_loss(Xe, ye):
        model.eval()
        log_probs = torch.log_softmax(model(Xe), dim=1)
        return criterion(log_probs, ye).item()

    for ep in range(1, max_epochs + 1):
        model.train()

        perm_tr = torch.randperm(X_tr.shape[0], device=device)
        Xs = X_tr[perm_tr]
        ys = y_tr[perm_tr]

        total = 0.0
        for start in range(0, Xs.shape[0], batch_size):
            xb = Xs[start:start + batch_size]
            yb = ys[start:start + batch_size]

            logits = model(xb)
            log_probs = torch.log_softmax(logits, dim=1)
            loss = criterion(log_probs, yb)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            total += loss.item() * xb.shape[0]

        train_loss = total / Xs.shape[0]
        val_loss = eval_loss(X_val, y_val)

        if (best_val - val_loss) > min_delta:
            best_val = val_loss
            best_state = copy.deepcopy(model.state_dict())
            best_epoch = ep
            bad_epochs = 0
        else:
            bad_epochs += 1

        if verbose and (ep == 1 or ep % print_every == 0):
            print(
                f"Epoch {ep}/{max_epochs} | train={train_loss:.6f} | val={val_loss:.6f} "
                f"| best_val={best_val:.6f} (ep {best_epoch}) | bad={bad_epochs}/{patience} | device={device}"
            )

        if bad_epochs >= patience:
            if verbose:
                print(f"Early stopping at epoch {ep}. Best val={best_val:.6f} at epoch {best_epoch}.")
            break

    model.load_state_dict(best_state)
    return model, {"best_epoch": best_epoch, "best_val": best_val, "device": device}


@torch.no_grad()
def eval_kl(model, X, y):
    model.eval()
    device = next(model.parameters()).device
    X = X.to(device)
    y = y.to(device)
    criterion = nn.KLDivLoss(reduction="batchmean")
    log_probs = torch.log_softmax(model(X), dim=1)
    return criterion(log_probs, y).item()


# -----------------------
# 3) Build Shaffer targets within one first-treatment stratum
# -----------------------
def build_targets_from_future_shaffer_stratified(
    X,
    adata,
    first_treatment,                 # one of: "cis", "cocl2", "dabtram"
    lineage_key="clone_id",
    condition_key="OG_condition_name",
    alpha_smooth=1e-3,
    drop_missing_future=True,
):
    """
    For one fixed first-treatment stratum:
      early inputs  = cells with condition == first_treatment
      future targets = lineage-level composition over the 3 two-treatment outcomes:
                       first_treatment -> {cis, cocl2, dabtram}
    """
    adata = adata.copy()
    cond = adata.obs[condition_key].astype(str)

    early_condition = first_treatment
    future_categories = [
        f"{first_treatment}tocis",
        f"{first_treatment}tococl2",
        f"{first_treatment}todabtram",
    ]

    # Future cells only from this first-treatment branch
    is_future = cond.isin(future_categories).to_numpy()
    adata_future = adata[is_future].copy()

    # clone -> 3-dim composition vector
    clone_to_probs = {}
    for clone_id, df in adata_future.obs.groupby(lineage_key):
        counts = np.array(
            [(df[condition_key].astype(str) == cat).sum() for cat in future_categories],
            dtype=float
        )
        counts = counts + alpha_smooth
        probs = counts / counts.sum()
        clone_to_probs[clone_id] = probs

    # Early cells from the fixed first-treatment stratum
    is_early = (cond.to_numpy() == early_condition)
    early_idx = np.where(is_early)[0]

    X_early = X[early_idx]
    clone_early = adata.obs.iloc[early_idx][lineage_key].to_numpy()

    y_prob = np.zeros((X_early.shape[0], 3), dtype=float)
    keep = np.ones(X_early.shape[0], dtype=bool)

    for i, cid in enumerate(clone_early):
        if cid in clone_to_probs:
            y_prob[i] = clone_to_probs[cid]
        else:
            if drop_missing_future:
                keep[i] = False
            else:
                y_prob[i] = np.ones(3) / 3.0

    X_early = X_early[keep]
    y_prob = y_prob[keep]
    y_prob = y_prob / y_prob.sum(axis=1, keepdims=True)

    return (
        torch.tensor(X_early, dtype=torch.float32),
        torch.tensor(y_prob, dtype=torch.float32),
        future_categories,
    )


# -----------------------
# 4) Optional filter by future clone size within one stratum
# -----------------------
def filter_by_clone_future_size_shaffer_stratified(
    adata,
    X,
    first_treatment,
    lineage_key="clone_id",
    condition_key="OG_condition_name",
    min_future_cells=10,
):
    """
    Keep all cells whose clone has >= min_future_cells among the future cells
    in the chosen first-treatment stratum.
    """
    cond = adata.obs[condition_key].astype(str)

    future_categories = [
        f"{first_treatment}tocis",
        f"{first_treatment}tococl2",
        f"{first_treatment}todabtram",
    ]

    is_future = cond.isin(future_categories).to_numpy()
    counts = adata.obs.loc[is_future, lineage_key].value_counts()
    keep_clones = set(counts[counts >= int(min_future_cells)].index)

    keep_mask = adata.obs[lineage_key].isin(keep_clones).to_numpy()
    return adata[keep_mask].copy(), X[keep_mask]


# -----------------------
# 5) One Shaffer KL experiment for one first-treatment stratum
# -----------------------
def run_one_threshold_experiment_shaffer_stratified(
    adata_train, X_train,
    adata_test, X_test,
    first_treatment,                 # "cis", "cocl2", or "dabtram"
    lineage_threshold=10,
    lineage_key="clone_id",
    condition_key="OG_condition_name",
    alpha_smooth=1e-3,
    device="mps",
    seed=42,
    lr=5e-3,
    weight_decay=1e-4,
    max_epochs=5000,
    batch_size=256,
    val_frac=0.2,
    patience=150,
    min_delta=1e-5,
    print_every=50,
):
    # 1) filter within split, no leakage
    ad_tr_f, X_tr_f = filter_by_clone_future_size_shaffer_stratified(
        adata_train, X_train,
        first_treatment=first_treatment,
        lineage_key=lineage_key,
        condition_key=condition_key,
        min_future_cells=lineage_threshold,
    )
    ad_te_f, X_te_f = filter_by_clone_future_size_shaffer_stratified(
        adata_test, X_test,
        first_treatment=first_treatment,
        lineage_key=lineage_key,
        condition_key=condition_key,
        min_future_cells=lineage_threshold,
    )

    # 2) build pairs
    X_tr2, y_tr, future_categories = build_targets_from_future_shaffer_stratified(
        X_tr_f, ad_tr_f,
        first_treatment=first_treatment,
        lineage_key=lineage_key,
        condition_key=condition_key,
        alpha_smooth=alpha_smooth,
    )
    X_te2, y_te, _ = build_targets_from_future_shaffer_stratified(
        X_te_f, ad_te_f,
        first_treatment=first_treatment,
        lineage_key=lineage_key,
        condition_key=condition_key,
        alpha_smooth=alpha_smooth,
    )

    print("First treatment:", first_treatment)
    print("Future categories:", future_categories)
    print("Train early cells used:", X_tr2.shape[0])
    print("Test early cells used:", X_te2.shape[0])

    # 3) train decoder
    model = LinearSoftmax(input_size=X_tr2.shape[1], output_size=3)
    model, hist = train_kl_earlystop(
        model,
        X_tr2, y_tr,
        lr=lr,
        weight_decay=weight_decay,
        max_epochs=max_epochs,
        batch_size=batch_size,
        val_frac=val_frac,
        patience=patience,
        min_delta=min_delta,
        seed=seed,
        device=device,
        print_every=print_every,
        verbose=True,
    )

    # 4) eval
    kl_train = eval_kl(model, X_tr2, y_tr)
    kl_test = eval_kl(model, X_te2, y_te)

    summary = {
        "first_treatment": first_treatment,
        "future_categories": future_categories,
        "min_future_cells": lineage_threshold,
        "train_cells_total_after_filter": ad_tr_f.n_obs,
        "test_cells_total_after_filter": ad_te_f.n_obs,
        "train_time1_cells_used": X_tr2.shape[0],
        "test_time1_cells_used": X_te2.shape[0],
        "best_epoch": hist["best_epoch"],
        "val_KL_best": hist["best_val"],
        "train_KL": kl_train,
        "test_KL": kl_test,
        "device": hist["device"],
    }

    print(summary)
    return model, summary

In [14]:
model_cis, summary_cis = run_one_threshold_experiment_shaffer_stratified(
    adata_train, X_train,
    adata_test, X_test,
    first_treatment="cis",
    lineage_threshold=10,
    lineage_key="clone_id",
    condition_key="OG_condition_name",
    device="mps",
    seed=42,
)

First treatment: cis
Future categories: ['cistocis', 'cistococl2', 'cistodabtram']
Train early cells used: 479
Test early cells used: 32
Epoch 1/5000 | train=0.780808 | val=0.629185 | best_val=0.629185 (ep 1) | bad=0/150 | device=mps
Epoch 50/5000 | train=0.143422 | val=0.171112 | best_val=0.171112 (ep 50) | bad=0/150 | device=mps
Epoch 100/5000 | train=0.103428 | val=0.125990 | best_val=0.125990 (ep 100) | bad=0/150 | device=mps
Epoch 150/5000 | train=0.085970 | val=0.099011 | best_val=0.099011 (ep 150) | bad=0/150 | device=mps
Epoch 200/5000 | train=0.075306 | val=0.083762 | best_val=0.083762 (ep 200) | bad=0/150 | device=mps
Epoch 250/5000 | train=0.068007 | val=0.073705 | best_val=0.073705 (ep 250) | bad=0/150 | device=mps
Epoch 300/5000 | train=0.062721 | val=0.067410 | best_val=0.067410 (ep 300) | bad=0/150 | device=mps
Epoch 350/5000 | train=0.058817 | val=0.065021 | best_val=0.065021 (ep 350) | bad=0/150 | device=mps
Epoch 400/5000 | train=0.055799 | val=0.062767 | best_val=0.0

In [15]:
model_cocl2, summary_cocl2 = run_one_threshold_experiment_shaffer_stratified(
    adata_train, X_train,
    adata_test, X_test,
    first_treatment="cocl2",
    lineage_threshold=10,
    lineage_key="clone_id",
    condition_key="OG_condition_name",
    device="mps",
    seed=42,
)



First treatment: cocl2
Future categories: ['cocl2tocis', 'cocl2tococl2', 'cocl2todabtram']
Train early cells used: 694
Test early cells used: 10
Epoch 1/5000 | train=0.388325 | val=0.329375 | best_val=0.329375 (ep 1) | bad=0/150 | device=mps
Epoch 50/5000 | train=0.040765 | val=0.053615 | best_val=0.053615 (ep 50) | bad=0/150 | device=mps
Epoch 100/5000 | train=0.027741 | val=0.040850 | best_val=0.040850 (ep 100) | bad=0/150 | device=mps
Epoch 150/5000 | train=0.023997 | val=0.035304 | best_val=0.035304 (ep 150) | bad=0/150 | device=mps
Epoch 200/5000 | train=0.021843 | val=0.032609 | best_val=0.032558 (ep 199) | bad=1/150 | device=mps
Epoch 250/5000 | train=0.020572 | val=0.031068 | best_val=0.030865 (ep 246) | bad=4/150 | device=mps
Epoch 300/5000 | train=0.019574 | val=0.029221 | best_val=0.029221 (ep 300) | bad=0/150 | device=mps
Epoch 350/5000 | train=0.018867 | val=0.028184 | best_val=0.027733 (ep 346) | bad=4/150 | device=mps
Epoch 400/5000 | train=0.018450 | val=0.028347 | best

In [16]:
model_dabtram, summary_dabtram = run_one_threshold_experiment_shaffer_stratified(
    adata_train, X_train,
    adata_test, X_test,
    first_treatment="dabtram",
    lineage_threshold=10,
    lineage_key="clone_id",
    condition_key="OG_condition_name",
    device="mps",
    seed=42,
)

First treatment: dabtram
Future categories: ['dabtramtocis', 'dabtramtococl2', 'dabtramtodabtram']
Train early cells used: 392
Test early cells used: 15
Epoch 1/5000 | train=0.620896 | val=0.539892 | best_val=0.539892 (ep 1) | bad=0/150 | device=mps
Epoch 50/5000 | train=0.085954 | val=0.081510 | best_val=0.081510 (ep 50) | bad=0/150 | device=mps
Epoch 100/5000 | train=0.067502 | val=0.059370 | best_val=0.059370 (ep 100) | bad=0/150 | device=mps
Epoch 150/5000 | train=0.059525 | val=0.049695 | best_val=0.049695 (ep 150) | bad=0/150 | device=mps
Epoch 200/5000 | train=0.055298 | val=0.043937 | best_val=0.043937 (ep 200) | bad=0/150 | device=mps
Epoch 250/5000 | train=0.052455 | val=0.040925 | best_val=0.040719 (ep 244) | bad=6/150 | device=mps
Epoch 300/5000 | train=0.050526 | val=0.038842 | best_val=0.038842 (ep 300) | bad=0/150 | device=mps
Epoch 350/5000 | train=0.049074 | val=0.037838 | best_val=0.037012 (ep 342) | bad=8/150 | device=mps
Epoch 400/5000 | train=0.048179 | val=0.03655